# Lab 3: Create and Organize Objects in Unity Catalog — TTC Open Data

This is my own comprehensive lab for Unit 5 (**Create and organize objects in Unity Catalog**), continuing the TTC open data theme from Labs 1 and 2. It's a **practice notebook**: each exercise explains a concept and gives you a task — write the SQL/Python yourself in the `# TODO` cells. Data-generation boilerplate (downloading files, faker records) is provided since that's not a Unity Catalog concept.

**Full checklist covered in this notebook:** naming conventions, catalog, schema, managed volume, managed table with PK/FK, external table, four function types (SQL scalar, Python, table-valued, temporary), a stored procedure, standard view, dynamic view (permissions), materialized view, streaming table, `ALTER TABLE` (add/rename/drop/retype), `ALTER CATALOG` (tags/ownership/predictive optimization), and `GRANT` permissions.

> Attach this notebook to **Serverless** compute. Unity Catalog must be enabled (default on new workspaces).

## Exercise 1: Naming Conventions and Catalog Creation

**Concept:** Unity Catalog names are lowercase with underscores; periods/spaces aren't allowed (periods separate the three-level namespace). A medallion layout uses `bronze` (raw), `silver` (cleaned), `gold` (aggregated) schemas within one catalog.

**Task:** Create a catalog named `ttc_open_data` with a descriptive comment. Then create three schemas inside it — `bronze`, `silver`, `gold` — each with its own comment.

> 🤖 **Genie Code tip:** *"Write SQL to create a Unity Catalog catalog with a comment, then three schemas inside it named bronze, silver, and gold, each with their own comment."*

In [ ]:
%sql
-- TODO: CREATE CATALOG IF NOT EXISTS ttc_open_data with a comment

-- TODO: CREATE SCHEMA IF NOT EXISTS for bronze, silver, and gold, each with a comment

## Exercise 2: Create a Volume

**Concept:** Volumes govern files (not tables) inside a schema — `catalog.schema.volume`. Managed volumes let Unity Catalog handle storage automatically; external volumes point at storage you already control.

**Task:** Create a **managed** volume named `landing_files` in `ttc_open_data.bronze`, with a comment describing it as a landing area for raw TTC files.

> 🤖 **Genie Code tip:** *"Write SQL to create a managed Unity Catalog volume named landing_files in a bronze schema."*

In [ ]:
%sql
-- TODO: CREATE VOLUME IF NOT EXISTS ttc_open_data.bronze.landing_files with a comment

## Exercise 3: Managed Table from a Real Dataset (with a Primary Key)

First, some provided boilerplate: download the real **TTC Routes and Schedules** GTFS feed and land `routes.txt` in your new volume. This part isn't a Unity Catalog concept, so it's filled in for you.

In [ ]:
import requests
import zipfile
import io

gtfs_url = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
    "7795b45e-e65a-4465-81fc-c36b9dfff169/resource/"
    "cfb6b2b8-6191-41e3-bda1-b175c51148cb/download/opendata_ttc_schedules.zip"
)

response = requests.get(gtfs_url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("routes.txt") as f:
        routes_csv_bytes = f.read()

volume_path = "/Volumes/ttc_open_data/bronze/landing_files"
dbutils.fs.put(f"{volume_path}/routes.txt", routes_csv_bytes.decode("utf-8"), overwrite=True)
print(f"Saved to {volume_path}/routes.txt")

**Concept:** Managed tables are the Unity Catalog default — Delta format, ACID transactions, automatic lifecycle management. Primary/foreign keys in Unity Catalog are informational (not enforced) but help documentation and the query optimizer.

**Task:**
1. Create (or replace) a managed table `ttc_open_data.bronze.raw_routes` using `CREATE TABLE ... AS SELECT` reading the CSV from the volume with `read_files(...)`. Select `route_id`, `route_short_name`, `route_long_name`, and `route_type` (cast to `INT`).
2. Add a primary key constraint on `route_id` using `ALTER TABLE ... ADD CONSTRAINT ... PRIMARY KEY (...)`.

> 🤖 **Genie Code tip:** *"Write a CREATE TABLE AS SELECT statement in Databricks SQL that reads a CSV file from a Unity Catalog volume using read_files, then show me how to add a primary key constraint to the resulting table with ALTER TABLE."*

In [ ]:
%sql
-- TODO: CREATE OR REPLACE TABLE ttc_open_data.bronze.raw_routes AS SELECT ... FROM read_files('/Volumes/ttc_open_data/bronze/landing_files/routes.txt', format => 'csv', header => true)

-- TODO: ALTER TABLE ttc_open_data.bronze.raw_routes ADD CONSTRAINT raw_routes_pk PRIMARY KEY (route_id)

## Exercise 4 (Optional/Advanced): External Table

**Concept:** External tables register metadata in Unity Catalog but leave the data files in their original location — dropping the table doesn't delete the files. They require a Unity Catalog **external location** (a cloud storage path + storage credential) to already be configured, pointing at the path you use.

**Prerequisite check:** Run `SHOW EXTERNAL LOCATIONS;` first. If you don't have one configured (this requires an Azure storage account + credential set up by a metastore admin — covered conceptually back in the Architecture unit), skip this exercise; it's here for completeness, not required to finish the lab.

**Task (if you have an external location):** Create an external table `ttc_open_data.bronze.raw_routes_external` pointing at a path within your external location, over the same `routes.txt` — using `CREATE EXTERNAL TABLE ... LOCATION ...`.

> 🤖 **Genie Code tip:** *"Write SQL to create an external table over a CSV file at a given external storage location in Unity Catalog."*

In [ ]:
%sql
-- Check for an available external location first:
SHOW EXTERNAL LOCATIONS;

-- TODO (if you have one): CREATE EXTERNAL TABLE ttc_open_data.bronze.raw_routes_external
-- USING CSV LOCATION '<your-external-location-path>/routes/' OPTIONS (header 'true', inferSchema 'true')

## Exercise 5: Managed Table with a Foreign Key

Provided boilerplate: generate 100 synthetic complaint records with `faker`, using real `route_id` values from `bronze.raw_routes`.

In [ ]:
%pip install faker==40.8.0

In [ ]:
import random
from faker import Faker

fake = Faker("en_CA")

real_route_ids = [row.route_id for row in spark.sql(
    "SELECT route_id FROM ttc_open_data.bronze.raw_routes"
).collect()]

issue_types = ["Delay", "Overcrowding", "Mechanical Fault", "Missed Stop", "Rude Operator"]

records = []
for complaint_id in range(1, 101):
    records.append({
        "complaint_id": complaint_id,
        "reporter_name": fake.name(),
        "filed_date": str(fake.date_between(start_date="2023-01-01", end_date="2025-12-31")),
        "route_id": random.choice(real_route_ids),
        "issue_type": random.choice(issue_types),
    })

complaints_df = spark.createDataFrame(records)
complaints_df.createOrReplaceTempView("complaints_staging")
display(complaints_df.limit(10))

**Task:**
1. Create a managed table `ttc_open_data.silver.ttc_complaints` with columns `complaint_id BIGINT NOT NULL`, `reporter_name STRING`, `filed_date DATE`, `route_id STRING`, `issue_type STRING`. Add a `PRIMARY KEY` on `complaint_id` and a `FOREIGN KEY` on `route_id` referencing `bronze.raw_routes(route_id)` — both as named `CONSTRAINT`s in the `CREATE TABLE` statement itself (not `ALTER TABLE` this time).
2. Insert the staged data from the `complaints_staging` temp view into the new table.

> 🤖 **Genie Code tip:** *"Write a CREATE TABLE statement in Databricks SQL with a primary key and a foreign key referencing another table, defined inline as named constraints, then an INSERT INTO ... SELECT from a temp view."*

In [ ]:
%sql
-- TODO: CREATE TABLE IF NOT EXISTS ttc_open_data.silver.ttc_complaints (...) with inline PK + FK constraints

-- TODO: INSERT INTO ttc_open_data.silver.ttc_complaints SELECT ... FROM complaints_staging

## Exercise 6: Functions — SQL Scalar, Python, Table-Valued, Temporary

**Concept:** Unity Catalog functions are governed, reusable objects. SQL scalar functions return a single value; Python functions handle more complex logic; table-valued functions return a full result set; temporary functions exist only for your session.

### 6a. SQL scalar function
**Task:** Create `ttc_open_data.gold.classify_issue_severity(issue_type STRING) RETURNS STRING` that returns `'High'` for `'Mechanical Fault'` or `'Missed Stop'`, else `'Low'`. Then query it against `silver.ttc_complaints`.

> 🤖 **Genie Code tip:** *"Write a Unity Catalog SQL scalar function using CREATE FUNCTION that classifies a string input into two categories with a CASE expression."*

In [ ]:
%sql
-- TODO: CREATE OR REPLACE FUNCTION ttc_open_data.gold.classify_issue_severity(issue_type STRING) RETURNS STRING ...

-- TODO: SELECT issue_type, ttc_open_data.gold.classify_issue_severity(issue_type) AS severity FROM ttc_open_data.silver.ttc_complaints LIMIT 10

### 6b. Python function
**Task:** Create `ttc_open_data.gold.normalize_name(input_text STRING) RETURNS STRING LANGUAGE PYTHON` that lowercases and strips a name (mirrors the `normalize_text` pattern from the unit). Test it on `reporter_name`.

> 🤖 **Genie Code tip:** *"Write a Unity Catalog Python function using CREATE FUNCTION ... LANGUAGE PYTHON that lowercases and strips whitespace from a string input."*

In [ ]:
%sql
-- TODO: CREATE OR REPLACE FUNCTION ttc_open_data.gold.normalize_name(input_text STRING) RETURNS STRING LANGUAGE PYTHON AS $$ ... $$

-- TODO: test it: SELECT reporter_name, ttc_open_data.gold.normalize_name(reporter_name) FROM ttc_open_data.silver.ttc_complaints LIMIT 5

### 6c. Table-valued function
**Task:** Create `ttc_open_data.gold.complaints_since(min_date DATE) RETURNS TABLE` that returns all complaints filed on or after `min_date`. Query it like a table with `SELECT * FROM ttc_open_data.gold.complaints_since(DATE '2024-06-01')`.

> 🤖 **Genie Code tip:** *"Write a Unity Catalog table-valued SQL function that takes a date parameter and returns rows from a table filtered by that date."*

In [ ]:
%sql
-- TODO: CREATE OR REPLACE FUNCTION ttc_open_data.gold.complaints_since(min_date DATE) RETURNS TABLE(...) RETURN SELECT ... WHERE filed_date >= min_date

-- TODO: SELECT * FROM ttc_open_data.gold.complaints_since(DATE '2024-06-01') LIMIT 10

### 6d. Temporary function
**Task:** Create a `TEMPORARY` function `days_since_filed(filed_date DATE) RETURNS INT` that computes `DATEDIFF(current_date(), filed_date)`. Remember: temporary functions don't take a catalog/schema qualifier.

> 🤖 **Genie Code tip:** *"Write a CREATE TEMPORARY FUNCTION in Databricks SQL that calculates the number of days between a date column and today."*

In [ ]:
%sql
-- TODO: CREATE TEMPORARY FUNCTION days_since_filed(filed_date DATE) RETURNS INT RETURN ...

-- TODO: test it against silver.ttc_complaints

## Exercise 7: Stored Procedure

**Concept:** Procedures orchestrate multi-step SQL with variables and control flow — requires Unity Catalog and Databricks Runtime 17.0+. `IN` parameters take input, `OUT` parameters return output, and `SQL SECURITY INVOKER` runs the procedure with the caller's own permissions.

**Task:** Create a procedure `ttc_open_data.silver.mark_old_complaints_resolved(IN cutoff_date DATE, OUT rows_updated INT)` that sets `resolved = true` for every complaint filed before `cutoff_date`, then sets `rows_updated` to the number of rows affected. Call it with `CALL ... (DATE '2024-01-01', :rows)`.

> 🤖 **Genie Code tip:** *"Write a Unity Catalog SQL stored procedure with an IN date parameter and an OUT integer parameter that updates rows in a table matching the date filter and returns the count of rows updated."*

> Note: this exercise assumes `resolved` already exists as a column (added in Exercise 12 below) — either do Exercise 12 first, or add the column here before running the procedure.

In [ ]:
%sql
-- TODO: CREATE PROCEDURE ttc_open_data.silver.mark_old_complaints_resolved(IN cutoff_date DATE, OUT rows_updated INT)
-- LANGUAGE SQL SQL SECURITY INVOKER BEGIN ... END;

-- TODO: CALL ttc_open_data.silver.mark_old_complaints_resolved(DATE '2024-01-01', :rows)

## Exercise 8: Standard View

**Task:** Create a view `ttc_open_data.silver.vw_complaints_with_route` joining `ttc_complaints` with `raw_routes` on `route_id`, selecting complaint details plus `route_short_name` and `route_long_name`.

> 🤖 **Genie Code tip:** *"Write a CREATE VIEW statement in Databricks SQL that joins two tables and selects specific columns from each."*

In [ ]:
%sql
-- TODO: CREATE OR REPLACE VIEW ttc_open_data.silver.vw_complaints_with_route AS SELECT ... JOIN ...

## Exercise 9: Dynamic View for Permissions

**Concept:** Dynamic views apply row/column security based on the querying user, using `is_account_group_member('<group>')` inside a `CASE` expression.

**Task:** Create `ttc_open_data.gold.vw_complaints_secure` that shows `reporter_name` only to members of a group called `ttc-auditors`; everyone else sees `'REDACTED'`. Base it on `vw_complaints_with_route`.

> 🤖 **Genie Code tip:** *"Write a Databricks SQL dynamic view using is_account_group_member() in a CASE statement to redact a column for users not in a specific group."*

**To actually test the masking:** create an account group `ttc-auditors` in your workspace's admin console, query the view while not a member (expect `REDACTED`), then add yourself and query again (expect real names).

In [ ]:
%sql
-- TODO: CREATE OR REPLACE VIEW ttc_open_data.gold.vw_complaints_secure AS SELECT ... CASE WHEN is_account_group_member('ttc-auditors') THEN reporter_name ELSE 'REDACTED' END AS reporter_name, ...

## Exercise 10: Materialized View with Incremental Refresh

**Concept:** Materialized views precompute and cache results as a Delta table. Incremental refresh (only processing changed data) requires row tracking enabled on the source table.

**Task:**
1. Enable row tracking on `ttc_open_data.silver.ttc_complaints` via `ALTER TABLE ... SET TBLPROPERTIES (delta.enableRowTracking = true)`.
2. Create a materialized view `ttc_open_data.gold.route_complaint_summary` that counts complaints per route (join in `route_short_name`/`route_long_name`), scheduled to refresh `EVERY 1 DAY`.

> 🤖 **Genie Code tip:** *"Write SQL to enable Delta row tracking on a table, then write a CREATE MATERIALIZED VIEW with a daily SCHEDULE that aggregates counts grouped by two columns."*

In [ ]:
%sql
-- TODO: ALTER TABLE ttc_open_data.silver.ttc_complaints SET TBLPROPERTIES (delta.enableRowTracking = true)

-- TODO: CREATE MATERIALIZED VIEW ttc_open_data.gold.route_complaint_summary SCHEDULE EVERY 1 DAY AS SELECT ... GROUP BY ...

## Exercise 11: Streaming Table

**Concept:** Streaming tables process only new rows that arrive after the last refresh — ideal for append-only ingestion (landing files, Kafka, medallion bronze/silver layers). Unlike materialized views, they don't reprocess old data on refresh.

Provided boilerplate: simulate a *second batch* of complaints landing as a new file, to give the streaming table something incremental to pick up.

In [ ]:
batch_2_records = []
for complaint_id in range(101, 121):
    batch_2_records.append({
        "complaint_id": complaint_id,
        "reporter_name": fake.name(),
        "filed_date": str(fake.date_between(start_date="2026-01-01", end_date="2026-06-30")),
        "route_id": random.choice(real_route_ids),
        "issue_type": random.choice(issue_types),
    })

batch_2_df = spark.createDataFrame(batch_2_records)
batch_2_pdf = batch_2_df.toPandas()
batch_2_csv = batch_2_pdf.to_csv(index=False)

dbutils.fs.put(
    "/Volumes/ttc_open_data/bronze/landing_files/complaints_batch_2.csv",
    batch_2_csv,
    overwrite=True,
)
print("Landed a second batch of complaints for the streaming table to pick up.")

**Task:** Create a streaming table `ttc_open_data.bronze.streaming_complaints` using `CREATE OR REFRESH STREAMING TABLE ... AS SELECT * FROM STREAM read_files(...)` pointed at the volume's CSV files, then trigger a refresh. Use `TRIGGER ON UPDATE` instead of a fixed schedule.

> 🤖 **Genie Code tip:** *"Write a CREATE OR REFRESH STREAMING TABLE statement in Databricks SQL that reads CSV files incrementally from a volume path using STREAM read_files, with TRIGGER ON UPDATE."*

In [ ]:
%sql
-- TODO: CREATE OR REFRESH STREAMING TABLE ttc_open_data.bronze.streaming_complaints
-- TRIGGER ON UPDATE
-- AS SELECT * FROM STREAM read_files('/Volumes/ttc_open_data/bronze/landing_files/', format => 'csv', header => true)

-- TODO: REFRESH STREAMING TABLE ttc_open_data.bronze.streaming_complaints

-- TODO: SELECT count(*) FROM ttc_open_data.bronze.streaming_complaints

## Exercise 12: `ALTER TABLE` — Add, Rename, Drop, Retype

**Task:** Against `ttc_open_data.silver.ttc_complaints`:
1. Add a `resolved BOOLEAN` column with a comment, then `UPDATE` it to `false` for all rows.
2. Rename `reporter_name` to `complainant_name` (requires enabling Delta column mapping first — check the unit's hint on `delta.columnMapping.mode`).
3. Add a throwaway column `temp_notes STRING`, then drop it.
4. Widen `complaint_id` from its current type to `BIGINT` if it isn't already (or pick another column to practice `ALTER COLUMN ... TYPE`).

> 🤖 **Genie Code tip:** *"Show me Databricks SQL ALTER TABLE examples for adding a column, enabling column mapping and renaming a column, dropping a column, and changing a column's type."*

In [ ]:
%sql
-- TODO: ALTER TABLE ... ADD COLUMN resolved BOOLEAN COMMENT '...'; then UPDATE ... SET resolved = false

-- TODO: ALTER TABLE ... SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); then ALTER TABLE ... RENAME COLUMN reporter_name TO complainant_name

-- TODO: ALTER TABLE ... ADD COLUMN temp_notes STRING; then ALTER TABLE ... DROP COLUMN temp_notes

-- TODO: ALTER TABLE ... ALTER COLUMN <col> TYPE <wider-type>

## Exercise 13: `ALTER CATALOG` — Tags, Ownership, Predictive Optimization

**Task:**
1. Apply tags to `ttc_open_data`: `domain = 'transit'`, `source = 'toronto_open_data'`.
2. Enable predictive optimization on the catalog.
3. (Optional — needs a real group/principal) Transfer ownership of the catalog to a group you manage.

> 🤖 **Genie Code tip:** *"Write ALTER CATALOG statements in Databricks SQL to set tags, enable predictive optimization, and transfer ownership to a group."*

In [ ]:
%sql
-- TODO: ALTER CATALOG ttc_open_data SET TAGS (domain = 'transit', source = 'toronto_open_data')

-- TODO: ALTER CATALOG ttc_open_data ENABLE PREDICTIVE OPTIMIZATION

-- TODO (optional): ALTER CATALOG ttc_open_data SET OWNER TO `<your-group>`

## Exercise 14: Grant Permissions

**Concept:** Unity Catalog permissions cascade — `USE CATALOG`/`USE SCHEMA` are needed to even see objects, then object-specific grants (`SELECT`, `READ VOLUME`, `WRITE VOLUME`, etc.) control what's usable.

**Task:** Grant, to a placeholder group `` `account users` `` (swap in a real group/your own identity to actually test this):
1. `USE CATALOG` on `ttc_open_data`
2. `USE SCHEMA` on the `gold` schema
3. `SELECT` on `gold.vw_complaints_secure` and `gold.route_complaint_summary`
4. `READ VOLUME` on `bronze.landing_files`
5. Bonus: also grant `WRITE VOLUME` to a *different* group to see the privilege split in Catalog Explorer

> 🤖 **Genie Code tip:** *"Write GRANT statements in Databricks SQL for USE CATALOG, USE SCHEMA, SELECT on a view, and READ VOLUME / WRITE VOLUME on a volume."*

In [ ]:
%sql
-- TODO: GRANT USE CATALOG ON CATALOG ttc_open_data TO `account users`

-- TODO: GRANT USE SCHEMA ON SCHEMA ttc_open_data.gold TO `account users`

-- TODO: GRANT SELECT ON VIEW ttc_open_data.gold.vw_complaints_secure TO `account users`
-- TODO: GRANT SELECT ON VIEW ttc_open_data.gold.route_complaint_summary TO `account users`

-- TODO: GRANT READ VOLUME ON VOLUME ttc_open_data.bronze.landing_files TO `account users`

## Recap

By the end of this notebook you should have practiced: naming conventions, catalog/schema/volume creation, a managed table with PK, an external table (if applicable), a managed table with PK+FK, four function types, a stored procedure, a standard view, a dynamic (permission-masking) view, a materialized view with row tracking, a streaming table, the full `ALTER TABLE` set, `ALTER CATALOG` (tags/predictive optimization/ownership), and `GRANT` permissions across catalog/schema/view/volume levels.

Next: **Lab 4** (foreign catalog over BigQuery) and **Lab 5** (AI/BI Genie space) — both build on the tables you just created here.